# K Ablation Study for Case Alignment

This notebook performs an ablation study of the k parameter for case alignment on tabular datasets.
It varies k from 3 to ~50 (odd numbers only) and visualizes how case alignment metrics change with k values.

## 1. Import Required Libraries

In [10]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.insert(0, os.path.abspath('../'))

from case_align.case_align import RobustnessCBR
from case_align.metrics import gower_distance_matrix

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Tabular Dataset

In [11]:
# Configuration: Choose which dataset to use
DATASET_NAME = "adult"  # Options: "adult", "bank", "beans", "cancer", "heloc", "mushroom", "ocean", "wine"
DATA_DIR = f"../data/{DATASET_NAME}"

# Load data
print(f"Loading {DATASET_NAME} dataset...")

X_train = torch.load(os.path.join(DATA_DIR, "Xtrain.pt"))
y_train = torch.load(os.path.join(DATA_DIR, "ytrain.pt"))
X_val = torch.load(os.path.join(DATA_DIR, "Xval.pt"))
y_val = torch.load(os.path.join(DATA_DIR, "yval.pt"))

# Convert to numpy
X_train = X_train.numpy() if isinstance(X_train, torch.Tensor) else X_train
y_train = y_train.numpy() if isinstance(y_train, torch.Tensor) else y_train
X_val = X_val.numpy() if isinstance(X_val, torch.Tensor) else X_val
y_val = y_val.numpy() if isinstance(y_val, torch.Tensor) else y_val


def to_class_labels(labels: np.ndarray) -> np.ndarray:
    labels = np.asarray(labels)
    if labels.ndim == 1:
        return labels
    if labels.ndim == 2:
        if labels.shape[1] == 1:
            return labels[:, 0]
        return np.argmax(labels, axis=1)
    raise ValueError(f"Unsupported label shape: {labels.shape}")


# For this study, use validation set as it's smaller and faster
X = np.asarray(X_val)
y = to_class_labels(y_val)

if X.shape[0] != y.shape[0]:
    raise ValueError(
        f"X/y row mismatch after label conversion: X has {X.shape[0]} rows, y has {y.shape[0]} rows. "
        f"Original y_val shape: {np.asarray(y_val).shape}"
    )

print(f"Dataset loaded: {DATASET_NAME}")
print(f"  Features shape: {X.shape}")
print(f"  Labels shape: {y.shape}")

unique, counts = np.unique(y.astype(int), return_counts=True)
class_dist = {int(k): int(v) for k, v in zip(unique, counts)}
print(f"  Class distribution: {class_dist}")
print(f"  Data type: {X.dtype}")

Loading adult dataset...
Dataset loaded: adult
  Features shape: (8045, 97)
  Labels shape: (8045,)
  Class distribution: {0: 6033, 1: 2012}
  Data type: float32


## 3. Generate or Load Explanations

In [12]:
# For demonstration, we'll use random explanations (normalized features)
# In practice, these would come from your explainer (e.g., Integrated Gradients, LIME, etc.)

np.random.seed(42)
explanations = np.random.randn(X.shape[0], X.shape[1])
explanations = (explanations - explanations.mean(axis=0)) / (explanations.std(axis=0) + 1e-8)

print(f"Explanations shape: {explanations.shape}")
print(f"Explanations stats:")
print(f"  Mean: {explanations.mean(axis=0)[:5]}")
print(f"  Std: {explanations.std(axis=0)[:5]}")
print(f"  Min: {explanations.min()}")
print(f"  Max: {explanations.max()}")

Explanations shape: (8045, 97)
Explanations stats:
  Mean: [-8.51124985e-18 -5.98582022e-18 -9.51176159e-18  5.94786978e-18
 -5.83608847e-17]
  Std: [0.99999999 0.99999999 0.99999999 0.99999999 0.99999999]
  Min: -4.822663071122065
  Max: 4.6671591947585185


## 4. Initialize Case Align Parameters and K Values

In [13]:
# Define k values: odd numbers from 3 to ~50
# Using sensible increments: by 2 from 3-21, then by 4 from 25-49
k_values = list(range(3, 22, 2)) + list(range(25, 50, 4))
k_values = sorted(set(k_values))  # Ensure sorted and unique

print(f"K values to test: {k_values}")
print(f"Number of k values: {len(k_values)}")

# Case align parameters
SIM_METRIC = "gower"  # "gower", "cosine", or "spearman"
PROBLEM_METRIC = "gower"
M_UNLIKE = 1
ROBUST_MODE = "geom"

# Identify feature types (all numeric for now, can be customized)
num_idx = np.arange(X.shape[1])
cat_idx = None

print(f"\nCase Align parameters:")
print(f"  sim_metric: {SIM_METRIC}")
print(f"  problem_metric: {PROBLEM_METRIC}")
print(f"  m_unlike: {M_UNLIKE}")
print(f"  robust_mode: {ROBUST_MODE}")

K values to test: [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 25, 29, 33, 37, 41, 45, 49]
Number of k values: 17

Case Align parameters:
  sim_metric: gower
  problem_metric: gower
  m_unlike: 1
  robust_mode: geom


## 5. Perform K Ablation Study

In [14]:
def compute_case_alignment_metrics(cbr, n_samples=None):
    """
    Compute case alignment metrics for the dataset.

    Args:
        cbr: RobustnessCBR instance (already fitted)
        n_samples: Number of samples to evaluate (None = all)

    Returns:
        dict with aggregated metrics
    """
    total = cbr.X.shape[0]
    n = total if n_samples is None else min(int(n_samples), total)
    indices = np.random.choice(total, size=n, replace=False)

    results = [cbr.compute_for_index(idx) for idx in indices]
    finite_ratios = [r.R_ratio for r in results if np.isfinite(r.R_ratio)]

    return {
        'S_plus_mean': np.mean([r.S_plus for r in results]),
        'S_plus_std': np.std([r.S_plus for r in results]),
        'S_minus_mean': np.mean([r.S_minus for r in results]),
        'S_minus_std': np.std([r.S_minus for r in results]),
        'R_ratio_mean': np.mean(finite_ratios) if len(finite_ratios) > 0 else np.nan,
        'R_ratio_std': np.std(finite_ratios) if len(finite_ratios) > 0 else np.nan,
        'R_bounded_mean': np.mean([r.R_bounded for r in results]),
        'R_bounded_std': np.std([r.R_bounded for r in results]),
    }


# Run ablation study
results_by_k = {}

# Set number of samples to evaluate per k (increase for more stable estimates)
N_SAMPLES_PER_K = 100

if X.shape[0] != y.shape[0]:
    raise ValueError(f"X/y mismatch before ablation: X has {X.shape[0]} rows but y has {y.shape[0]} rows")
if explanations.shape[0] != X.shape[0]:
    raise ValueError(
        f"Explanations/X mismatch before ablation: explanations has {explanations.shape[0]} rows but X has {X.shape[0]} rows"
    )

print("Running k ablation study...")
print(f"Evaluating {min(N_SAMPLES_PER_K, X.shape[0])} samples per k value")
print("-" * 60)

for i, k in enumerate(k_values):
    print(f"[{i+1}/{len(k_values)}] Testing k={k}...", end=" ", flush=True)

    try:
        # Create and fit case align model
        cbr = RobustnessCBR(
            k=k,
            m_unlike=M_UNLIKE,
            sim_metric=SIM_METRIC,
            problem_metric=PROBLEM_METRIC,
            num_idx=num_idx,
            cat_idx=cat_idx,
            robust_mode=ROBUST_MODE,
            random_state=42
        )
        cbr.fit(X, y, explanations)

        # Compute metrics (subset for speed)
        metrics = compute_case_alignment_metrics(cbr, n_samples=N_SAMPLES_PER_K)
        results_by_k[k] = metrics

        print(f"✓ S+={metrics['S_plus_mean']:.4f}, S-={metrics['S_minus_mean']:.4f}, R={metrics['R_bounded_mean']:.4f}")
    except Exception as e:
        print(f"✗ Error: {e}")
        results_by_k[k] = None

print("-" * 60)
print("Ablation study completed!")

# Convert to DataFrame for easier analysis
results_df = pd.DataFrame([
    {
        'k': k,
        'S_plus_mean': v['S_plus_mean'] if v else np.nan,
        'S_plus_std': v['S_plus_std'] if v else np.nan,
        'S_minus_mean': v['S_minus_mean'] if v else np.nan,
        'S_minus_std': v['S_minus_std'] if v else np.nan,
        'R_bounded_mean': v['R_bounded_mean'] if v else np.nan,
        'R_bounded_std': v['R_bounded_std'] if v else np.nan,
    }
    for k, v in results_by_k.items()
])

print("\nResults Summary:")
print(results_df.to_string(index=False))

Running k ablation study...
Evaluating 100 samples per k value
------------------------------------------------------------
[1/17] Testing k=3... ✓ S+=0.2029, S-=0.2114, R=0.3199
[2/17] Testing k=5... ✓ S+=0.2071, S-=0.2119, R=0.3240
[3/17] Testing k=7... ✓ S+=0.2081, S-=0.2097, R=0.3241
[4/17] Testing k=9... ✓ S+=0.2092, S-=0.2096, R=0.3248
[5/17] Testing k=11... ✓ S+=0.2087, S-=0.2103, R=0.3251
[6/17] Testing k=13... ✓ S+=0.2084, S-=0.2106, R=0.3247
[7/17] Testing k=15... ✓ S+=0.2084, S-=0.2101, R=0.3250
[8/17] Testing k=17... ✓ S+=0.2086, S-=0.2100, R=0.3247
[9/17] Testing k=19... ✓ S+=0.2087, S-=0.2096, R=0.3248
[10/17] Testing k=21... ✓ S+=0.2096, S-=0.2094, R=0.3253
[11/17] Testing k=25... ✓ S+=0.2097, S-=0.2096, R=0.3254
[12/17] Testing k=29... ✓ S+=0.2098, S-=0.2102, R=0.3259
[13/17] Testing k=33... ✓ S+=0.2095, S-=0.2100, R=0.3255
[14/17] Testing k=37... ✓ S+=0.2095, S-=0.2100, R=0.3255
[15/17] Testing k=41... ✓ S+=0.2093, S-=0.2096, R=0.3252
[16/17] Testing k=45... ✓ S+=0.209

## 6. Visualize Results

In [17]:
# Create less-squished visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("S+ (Like Case Alignment)", "S- (Unlike Case Alignment)",
                    "R_bounded (Robustness)", "S+ vs S- Comparison"),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]],
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

# Filter out NaN values
valid_results = results_df.dropna()

# Plot 1: S+ (Like case alignment)
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_plus_mean'],
        error_y=dict(
            type='data',
            array=valid_results['S_plus_std'],
            visible=True
        ),
        mode='lines+markers',
        name='S+ (alignment with like cases)',
        line=dict(color='green', width=2.5),
        marker=dict(size=7)
    ),
    row=1, col=1
)

# Plot 2: S- (Unlike case alignment)
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_minus_mean'],
        error_y=dict(
            type='data',
            array=valid_results['S_minus_std'],
            visible=True
        ),
        mode='lines+markers',
        name='S- (alignment with unlike cases)',
        line=dict(color='red', width=2.5),
        marker=dict(size=7)
    ),
    row=1, col=2
)

# Plot 3: R_bounded (Robustness)
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['R_bounded_mean'],
        error_y=dict(
            type='data',
            array=valid_results['R_bounded_std'],
            visible=True
        ),
        mode='lines+markers',
        name='R_bounded (robustness)',
        line=dict(color='blue', width=2.5),
        marker=dict(size=7)
    ),
    row=2, col=1
)

# Plot 4: S+ vs S- comparison
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_plus_mean'],
        mode='lines+markers',
        name='S+ (like)',
        line=dict(color='green', width=2.5, dash='solid'),
        marker=dict(size=7)
    ),
    row=2, col=2
)

fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_minus_mean'],
        mode='lines+markers',
        name='S- (unlike)',
        line=dict(color='red', width=2.5, dash='dash'),
        marker=dict(size=7)
    ),
    row=2, col=2
)

# Update axes labels
fig.update_xaxes(title_text="k (number of neighbors)", row=1, col=1)
fig.update_yaxes(title_text="S+ Score", row=1, col=1)

fig.update_xaxes(title_text="k (number of neighbors)", row=1, col=2)
fig.update_yaxes(title_text="S- Score", row=1, col=2)

fig.update_xaxes(title_text="k (number of neighbors)", row=2, col=1)
fig.update_yaxes(title_text="R_bounded Score", row=2, col=1)

fig.update_xaxes(title_text="k (number of neighbors)", row=2, col=2)
fig.update_yaxes(title_text="Alignment Score", row=2, col=2)

# Update layout for readability
fig.update_layout(
    title_text=f"K Ablation Study for Case Alignment (Dataset: {DATASET_NAME})",
    template='plotly_white',
    height=1250,
    width=1450,
    showlegend=True,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.06,
        xanchor='left',
        x=0.0
    ),
    margin=dict(l=80, r=40, t=170, b=80),
    font=dict(size=13)
)
fig.update_annotations(font=dict(size=14))

fig.show()

print("Visualization complete!")

Visualization complete!


## 7. Analysis and Insights

In [16]:
# Compute statistics
valid_results = results_df.dropna()

print("=" * 70)
print("K ABLATION STUDY ANALYSIS")
print("=" * 70)

print(f"\nDataset: {DATASET_NAME}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"K values tested: {len(valid_results)}")

print("\n" + "-" * 70)
print("S+ (Alignment with Like Cases)")
print("-" * 70)
idx_max_s_plus = valid_results['S_plus_mean'].idxmax()
print(f"Maximum at k={valid_results.loc[idx_max_s_plus, 'k']:.0f}: {valid_results.loc[idx_max_s_plus, 'S_plus_mean']:.4f}")
idx_min_s_plus = valid_results['S_plus_mean'].idxmin()
print(f"Minimum at k={valid_results.loc[idx_min_s_plus, 'k']:.0f}: {valid_results.loc[idx_min_s_plus, 'S_plus_mean']:.4f}")
print(f"Mean across k: {valid_results['S_plus_mean'].mean():.4f}")
print(f"Std across k: {valid_results['S_plus_mean'].std():.4f}")

print("\n" + "-" * 70)
print("S- (Alignment with Unlike Cases)")
print("-" * 70)
idx_max_s_minus = valid_results['S_minus_mean'].idxmax()
print(f"Maximum at k={valid_results.loc[idx_max_s_minus, 'k']:.0f}: {valid_results.loc[idx_max_s_minus, 'S_minus_mean']:.4f}")
idx_min_s_minus = valid_results['S_minus_mean'].idxmin()
print(f"Minimum at k={valid_results.loc[idx_min_s_minus, 'k']:.0f}: {valid_results.loc[idx_min_s_minus, 'S_minus_mean']:.4f}")
print(f"Mean across k: {valid_results['S_minus_mean'].mean():.4f}")
print(f"Std across k: {valid_results['S_minus_mean'].std():.4f}")

print("\n" + "-" * 70)
print("R_bounded (Robustness Score)")
print("-" * 70)
idx_max_r = valid_results['R_bounded_mean'].idxmax()
print(f"Maximum at k={valid_results.loc[idx_max_r, 'k']:.0f}: {valid_results.loc[idx_max_r, 'R_bounded_mean']:.4f}")
idx_min_r = valid_results['R_bounded_mean'].idxmin()
print(f"Minimum at k={valid_results.loc[idx_min_r, 'k']:.0f}: {valid_results.loc[idx_min_r, 'R_bounded_mean']:.4f}")
print(f"Mean across k: {valid_results['R_bounded_mean'].mean():.4f}")
print(f"Std across k: {valid_results['R_bounded_mean'].std():.4f}")

print("\n" + "-" * 70)
print("Key Observations:")
print("-" * 70)

# Check monotonicity
s_plus_trend = "increasing" if valid_results['S_plus_mean'].iloc[-1] > valid_results['S_plus_mean'].iloc[0] else "decreasing"
s_minus_trend = "increasing" if valid_results['S_minus_mean'].iloc[-1] > valid_results['S_minus_mean'].iloc[0] else "decreasing"

print(f"• S+ trend over k range: {s_plus_trend}")
print(f"• S- trend over k range: {s_minus_trend}")
print(f"• Best k for robustness (R_bounded): k={valid_results.loc[idx_max_r, 'k']:.0f}")
print(f"• Alignment variance (S+): {valid_results['S_plus_mean'].std():.4f}")
print(f"• Alignment variance (S-): {valid_results['S_minus_mean'].std():.4f}")

print("\n" + "=" * 70)

K ABLATION STUDY ANALYSIS

Dataset: adult
Number of samples: 8045
Number of features: 97
K values tested: 17

----------------------------------------------------------------------
S+ (Alignment with Like Cases)
----------------------------------------------------------------------
Maximum at k=29: 0.2098
Minimum at k=3: 0.2029
Mean across k: 0.2086
Std across k: 0.0016

----------------------------------------------------------------------
S- (Alignment with Unlike Cases)
----------------------------------------------------------------------
Maximum at k=5: 0.2119
Minimum at k=21: 0.2094
Mean across k: 0.2101
Std across k: 0.0007

----------------------------------------------------------------------
R_bounded (Robustness Score)
----------------------------------------------------------------------
Maximum at k=29: 0.3259
Minimum at k=3: 0.3199
Mean across k: 0.3247
Std across k: 0.0013

----------------------------------------------------------------------
Key Observations:
---------

In [ ]:
# Compute statistics
valid_results = results_df.dropna()

print("=" * 70)
print("K ABLATION STUDY ANALYSIS")
print("=" * 70)

print(f"\nDataset: {DATASET_NAME}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"K values tested: {len(valid_results)}")

print("\n" + "-" * 70)
print("S+ (Alignment with Like Cases)")
print("-" * 70)
idx_max_s_plus = valid_results['S_plus_mean'].idxmax()
print(f"Maximum at k={valid_results.loc[idx_max_s_plus, 'k']:.0f}: {valid_results.loc[idx_max_s_plus, 'S_plus_mean']:.4f}")
idx_min_s_plus = valid_results['S_plus_mean'].idxmin()
print(f"Minimum at k={valid_results.loc[idx_min_s_plus, 'k']:.0f}: {valid_results.loc[idx_min_s_plus, 'S_plus_mean']:.4f}")
print(f"Mean across k: {valid_results['S_plus_mean'].mean():.4f}")
print(f"Std across k: {valid_results['S_plus_mean'].std():.4f}")

print("\n" + "-" * 70)
print("S- (Alignment with Unlike Cases)")
print("-" * 70)
idx_max_s_minus = valid_results['S_minus_mean'].idxmax()
print(f"Maximum at k={valid_results.loc[idx_max_s_minus, 'k']:.0f}: {valid_results.loc[idx_max_s_minus, 'S_minus_mean']:.4f}")
idx_min_s_minus = valid_results['S_minus_mean'].idxmin()
print(f"Minimum at k={valid_results.loc[idx_min_s_minus, 'k']:.0f}: {valid_results.loc[idx_min_s_minus, 'S_minus_mean']:.4f}")
print(f"Mean across k: {valid_results['S_minus_mean'].mean():.4f}")
print(f"Std across k: {valid_results['S_minus_mean'].std():.4f}")

print("\n" + "-" * 70)
print("R_bounded (Robustness Score)")
print("-" * 70)
idx_max_r = valid_results['R_bounded_mean'].idxmax()
print(f"Maximum at k={valid_results.loc[idx_max_r, 'k']:.0f}: {valid_results.loc[idx_max_r, 'R_bounded_mean']:.4f}")
idx_min_r = valid_results['R_bounded_mean'].idxmin()
print(f"Minimum at k={valid_results.loc[idx_min_r, 'k']:.0f}: {valid_results.loc[idx_min_r, 'R_bounded_mean']:.4f}")
print(f"Mean across k: {valid_results['R_bounded_mean'].mean():.4f}")
print(f"Std across k: {valid_results['R_bounded_mean'].std():.4f}")

print("\n" + "-" * 70)
print("Key Observations:")
print("-" * 70)

# Check monotonicity
s_plus_trend = "increasing" if valid_results['S_plus_mean'].iloc[-1] > valid_results['S_plus_mean'].iloc[0] else "decreasing"
s_minus_trend = "increasing" if valid_results['S_minus_mean'].iloc[-1] > valid_results['S_minus_mean'].iloc[0] else "decreasing"

print(f"• S+ trend over k range: {s_plus_trend}")
print(f"• S- trend over k range: {s_minus_trend}")
print(f"• Best k for robustness (R_bounded): k={valid_results.loc[idx_max_r, 'k']:.0f}")
print(f"• Alignment variance (S+): {valid_results['S_plus_mean'].std():.4f}")
print(f"• Alignment variance (S-): {valid_results['S_minus_mean'].std():.4f}")

print("\n" + "=" * 70)

NameError: name 'results_df' is not defined

## 7. Analysis and Insights

In [ ]:
# Create less-squished visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("S+ (Like Case Alignment)", "S- (Unlike Case Alignment)",
                    "R_bounded (Robustness)", "S+ vs S- Comparison"),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]],
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

# Filter out NaN values
valid_results = results_df.dropna()

# Plot 1: S+ (Like case alignment)
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_plus_mean'],
        error_y=dict(
            type='data',
            array=valid_results['S_plus_std'],
            visible=True
        ),
        mode='lines+markers',
        name='S+ (alignment with like cases)',
        line=dict(color='green', width=2.5),
        marker=dict(size=7)
    ),
    row=1, col=1
)

# Plot 2: S- (Unlike case alignment)
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_minus_mean'],
        error_y=dict(
            type='data',
            array=valid_results['S_minus_std'],
            visible=True
        ),
        mode='lines+markers',
        name='S- (alignment with unlike cases)',
        line=dict(color='red', width=2.5),
        marker=dict(size=7)
    ),
    row=1, col=2
)

# Plot 3: R_bounded (Robustness)
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['R_bounded_mean'],
        error_y=dict(
            type='data',
            array=valid_results['R_bounded_std'],
            visible=True
        ),
        mode='lines+markers',
        name='R_bounded (robustness)',
        line=dict(color='blue', width=2.5),
        marker=dict(size=7)
    ),
    row=2, col=1
)

# Plot 4: S+ vs S- comparison
fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_plus_mean'],
        mode='lines+markers',
        name='S+ (like)',
        line=dict(color='green', width=2.5, dash='solid'),
        marker=dict(size=7)
    ),
    row=2, col=2
)

fig.add_trace(
    go.Scatter(
        x=valid_results['k'],
        y=valid_results['S_minus_mean'],
        mode='lines+markers',
        name='S- (unlike)',
        line=dict(color='red', width=2.5, dash='dash'),
        marker=dict(size=7)
    ),
    row=2, col=2
)

# Update axes labels
fig.update_xaxes(title_text="k (number of neighbors)", row=1, col=1)
fig.update_yaxes(title_text="S+ Score", row=1, col=1)

fig.update_xaxes(title_text="k (number of neighbors)", row=1, col=2)
fig.update_yaxes(title_text="S- Score", row=1, col=2)

fig.update_xaxes(title_text="k (number of neighbors)", row=2, col=1)
fig.update_yaxes(title_text="R_bounded Score", row=2, col=1)

fig.update_xaxes(title_text="k (number of neighbors)", row=2, col=2)
fig.update_yaxes(title_text="Alignment Score", row=2, col=2)

# Update layout for readability
fig.update_layout(
    title_text=f"K Ablation Study for Case Alignment (Dataset: {DATASET_NAME})",
    template='plotly_white',
    height=1250,
    width=1450,
    showlegend=True,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.06,
        xanchor='left',
        x=0.0
    ),
    margin=dict(l=80, r=40, t=170, b=80),
    font=dict(size=13)
)
fig.update_annotations(font=dict(size=14))

fig.show()

print("Visualization complete!")

## 6. Visualize Results

In [ ]:
def compute_case_alignment_metrics(cbr, n_samples=None):
    """
    Compute case alignment metrics for the dataset.
    
    Args:
        cbr: RobustnessCBR instance (already fitted)
        n_samples: Number of samples to evaluate (None = all)
    
    Returns:
        dict with aggregated metrics
    """
    n = n_samples if n_samples is not None else cbr.X.shape[0]
    indices = np.random.choice(cbr.X.shape[0], size=n, replace=False)
    
    results = [cbr.compute_for_index(idx) for idx in indices]
    
    return {
        'S_plus_mean': np.mean([r.S_plus for r in results]),
        'S_plus_std': np.std([r.S_plus for r in results]),
        'S_minus_mean': np.mean([r.S_minus for r in results]),
        'S_minus_std': np.std([r.S_minus for r in results]),
        'R_ratio_mean': np.mean([r.R_ratio for r in results if r.R_ratio != np.inf]),
        'R_ratio_std': np.std([r.R_ratio for r in results if r.R_ratio != np.inf]),
        'R_bounded_mean': np.mean([r.R_bounded for r in results]),
        'R_bounded_std': np.std([r.R_bounded for r in results]),
    }

# Run ablation study
results_by_k = {}

print("Running k ablation study...")
print("-" * 60)

for i, k in enumerate(k_values):
    print(f"[{i+1}/{len(k_values)}] Testing k={k}...", end=" ", flush=True)
    
    try:
        # Create and fit case align model
        cbr = RobustnessCBR(
            k=k,
            m_unlike=M_UNLIKE,
            sim_metric=SIM_METRIC,
            problem_metric=PROBLEM_METRIC,
            num_idx=num_idx,
            cat_idx=cat_idx,
            robust_mode=ROBUST_MODE,
            random_state=42
        )
        cbr.fit(X, y, explanations)
        
        # Compute metrics (using all samples for accuracy)
        metrics = compute_case_alignment_metrics(cbr, n_samples=None)
        results_by_k[k] = metrics
        
        print(f"✓ S+={metrics['S_plus_mean']:.4f}, S-={metrics['S_minus_mean']:.4f}, R={metrics['R_bounded_mean']:.4f}")
    except Exception as e:
        print(f"✗ Error: {e}")
        results_by_k[k] = None

print("-" * 60)
print("Ablation study completed!")

# Convert to DataFrame for easier analysis
results_df = pd.DataFrame([
    {
        'k': k,
        'S_plus_mean': v['S_plus_mean'] if v else np.nan,
        'S_plus_std': v['S_plus_std'] if v else np.nan,
        'S_minus_mean': v['S_minus_mean'] if v else np.nan,
        'S_minus_std': v['S_minus_std'] if v else np.nan,
        'R_bounded_mean': v['R_bounded_mean'] if v else np.nan,
        'R_bounded_std': v['R_bounded_std'] if v else np.nan,
    }
    for k, v in results_by_k.items()
])

print("\nResults Summary:")
print(results_df.to_string())

## 5. Perform K Ablation Study

In [ ]:
# Define k values: odd numbers from 3 to ~50
# Using sensible increments: by 2 from 3-21, then by 4 from 25-49
k_values = list(range(3, 22, 2)) + list(range(25, 50, 4))
k_values = sorted(set(k_values))  # Ensure sorted and unique

print(f"K values to test: {k_values}")
print(f"Number of k values: {len(k_values)}")

# Case align parameters
SIM_METRIC = "gower"  # "gower", "cosine", or "spearman"
PROBLEM_METRIC = "gower"
M_UNLIKE = 1
ROBUST_MODE = "geom"

# Identify feature types (all numeric for now, can be customized)
num_idx = np.arange(X.shape[1])
cat_idx = None

print(f"\nCase Align parameters:")
print(f"  sim_metric: {SIM_METRIC}")
print(f"  problem_metric: {PROBLEM_METRIC}")
print(f"  m_unlike: {M_UNLIKE}")
print(f"  robust_mode: {ROBUST_MODE}")

## 4. Initialize Case Align Parameters and K Values

In [ ]:
# For demonstration, we'll use random explanations (normalized features)
# In practice, these would come from your explainer (e.g., Integrated Gradients, LIME, etc.)

np.random.seed(42)
explanations = np.random.randn(X.shape[0], X.shape[1])
explanations = (explanations - explanations.mean(axis=0)) / (explanations.std(axis=0) + 1e-8)

print(f"Explanations shape: {explanations.shape}")
print(f"Explanations stats:")
print(f"  Mean: {explanations.mean(axis=0)[:5]}")
print(f"  Std: {explanations.std(axis=0)[:5]}")
print(f"  Min: {explanations.min()}")
print(f"  Max: {explanations.max()}")

## 3. Generate or Load Explanations

In [ ]:
# Configuration: Choose which dataset to use
DATASET_NAME = "adult"  # Options: "adult", "bank", "beans", "cancer", "heloc", "mushroom", "ocean", "wine"
DATA_DIR = f"../data/{DATASET_NAME}"

# Load data
print(f"Loading {DATASET_NAME} dataset...")

X_train = torch.load(os.path.join(DATA_DIR, "Xtrain.pt"))
y_train = torch.load(os.path.join(DATA_DIR, "ytrain.pt"))
X_val = torch.load(os.path.join(DATA_DIR, "Xval.pt"))
y_val = torch.load(os.path.join(DATA_DIR, "yval.pt"))

# Convert to numpy
X_train = X_train.numpy() if isinstance(X_train, torch.Tensor) else X_train
y_train = y_train.numpy() if isinstance(y_train, torch.Tensor) else y_train
X_val = X_val.numpy() if isinstance(X_val, torch.Tensor) else X_val
y_val = y_val.numpy() if isinstance(y_val, torch.Tensor) else y_val

# For this study, use validation set as it's smaller and faster
X = X_val
y = y_val

print(f"Dataset loaded: {DATASET_NAME}")
print(f"  Features shape: {X.shape}")
print(f"  Labels shape: {y.shape}")
print(f"  Class distribution: {np.bincount(y.astype(int))}")
print(f"  Data type: {X.dtype}")

## 2. Load Tabular Dataset

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.insert(0, os.path.abspath('../'))

from case_align.case_align import RobustnessCBR
from case_align.metrics import gower_distance_matrix

print("Libraries imported successfully!")

## 1. Import Required Libraries

# K Ablation Study for Case Alignment

This notebook performs an ablation study of the k parameter for case alignment on tabular datasets.
It varies k from 3 to ~50 (odd numbers only) and visualizes how case alignment metrics change with k values.